# Environment / Dependencies Setup
The cell below installs all required Python packages:

In [ ]:
!pip install -qU langchain langchain-classic langchain-community langchain-openai langchain-huggingface langchain-qdrant qdrant-client pypdf tiktoken flask numpy scipy scikit-learn sentence-transformers python-dotenv "torch>=2.5" ipython

### Step 1 — Imports

In [ ]:
import os

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.stores import InMemoryByteStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

### Step 2 — Configure Models

In [ ]:
# Load the secrets (QDRANT_URL, QDRANT_API_KEY, OPENROUTER_API_KEY) from .env
load_dotenv(".env")

# Initialize Chat Model
llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    # openai-compatible client, but routed to OpenRouter's servers
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0.0
)

# Initialize Dense Embedding Model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


### Step 3 — Initialize Storage Architecture or Use Existing if already made

In [ ]:
# Qdrant Cloud holds the child & summary VECTORS (the semantic search index)
vectorstore = QdrantVectorStore.from_texts(
    # throwaway seed text: from_texts needs at least one string to create the collection
    texts=["Initialize"], 
    embedding=embeddings, 
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    collection_name="multi_vector_collection"
)

# Will hold the full parent documents, keyed by doc_id, once the retriever wraps it
store = InMemoryByteStore()
id_key = "doc_id"


In [ ]:
# Use this instead of the cell above ONLY if you already ran this notebook before and the vectors already exist in Qdrant. Comment out the cell above and uncomment this one.
# vectorstore = QdrantVectorStore.from_existing_collection(
#     embedding=embeddings,
#     url=os.getenv("QDRANT_URL"),
#     api_key=os.getenv("QDRANT_API_KEY"),
#     collection_name="multi_vector_collection"
# )
#
# store = InMemoryByteStore()
# id_key = "doc_id"


In [ ]:
# OPTIONAL: no Qdrant Cloud account? Run this INSTEAD of the Qdrant Cloud cell above.
# It keeps the vectors in this notebook's memory, so they disappear when the kernel restarts.
vectorstore = QdrantVectorStore.from_texts(
    texts=["Initialize"],
    embedding=embeddings,
    location=":memory:",
    collection_name="multi_vector_collection"
)

store = InMemoryByteStore()
id_key = "doc_id"

In [ ]:
# OPTIONAL: delete the throwaway "Initialize" vector so it can never show up as a search hit.
# Run it once, right after the cell that creates the collection and before Step 8.
from qdrant_client import models

vectorstore.client.delete(
    collection_name="multi_vector_collection",
    points_selector=models.FilterSelector(
        filter=models.Filter(
            must=[models.FieldCondition(key="page_content", match=models.MatchValue(value="Initialize"))]
        )
    ),
)

### Step 4 — Download & Extract Document

In [ ]:
# Attention is All You Need paper (source document for this lab)
PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"
PDF_FILENAME = "attention_paper.pdf"

# Download the PDF and save it locally; raise an error on HTTP failure
response = requests.get(PDF_URL)
response.raise_for_status()

with open(PDF_FILENAME, "wb") as f:
    f.write(response.content)

In [ ]:
# Extract text from every page and join pages with newlines into one string
reader = PdfReader(PDF_FILENAME)
raw_text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])

print(f"Extracted {len(raw_text)} characters.")

### Step 5 — Generate Parent Documents

In [ ]:
# Parent chunks are LARGE (~10k chars) so the LLM has full context to answer from
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=200)
parent_chunks = parent_splitter.split_text(raw_text)

# Wrap each chunk in a Document (parents are stored, not embedded directly)
parent_docs = [Document(page_content=chunk) for chunk in parent_chunks]

print(f"Created {len(parent_docs)} Parent Documents.")

### Step 6 — Generate Child Documents
Each child gets a deterministic ID derived from its parent's text, linking parent to children.

In [ ]:
import hashlib

def stable_id(text):
    """Deterministic ID derived from the chunk text, so the same chunk always gets the same ID across runs."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

Child chunks are deliberately small, so they embed precisely for semantic search.

In [ ]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
child_docs = []

for doc in parent_docs:
    _id = stable_id(doc.page_content)
    # the parent carries its own id, which every child below will inherit
    doc.metadata["doc_id"] = _id

    child_splits = child_splitter.split_documents([doc])
    for child in child_splits:
        # stamp the parent id so retrieval can jump from child chunk back to parent
        child.metadata["doc_id"] = _id

    child_docs.extend(child_splits)

print(f"Created {len(child_docs)} Child Documents.")

### Step 7 — Define and Execute Summarization Pipeline

In [ ]:
# Prompt: compress a parent chunk into a short summary while keeping key terms
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize the following core concepts concisely. Do not lose technical keywords.\n\nText: {context}"
)

# LCEL chain: pass text through unchanged → prompt → LLM → plain string output
summary_chain = (
    {"context": RunnablePassthrough()} 
    | summary_prompt 
    | llm 
    | StrOutputParser()
)

In [ ]:
summary_docs = []

# Summarize each parent and carry its doc_id so the summary maps back to it
for doc in parent_docs:
    summary_text = summary_chain.invoke(doc.page_content)
    
    summary_doc = Document(
        page_content=summary_text,
        metadata={"doc_id": doc.metadata["doc_id"]}
    )
    summary_docs.append(summary_doc)

print(f"Generated {len(summary_docs)} Summaries.")

### Step 8 — Configure Multi-Vector Retriever

In [ ]:
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,   # searches child/summary embeddings
    byte_store=store,          # returns the linked parent documents
    id_key=id_key,             # the metadata field linking vectors to parents
    search_kwargs={"k": 2}    # return top-2 matches per query
)

# Store the full parent Documents in the parent store (auto-serialized into the byte store)
retriever.docstore.mset([(doc.metadata[id_key], doc) for doc in parent_docs])

# First-time setup: uploads vectors to Qdrant. If you already ran this notebook before and data already exists in Qdrant, comment out these two lines to avoid uploading duplicates.
retriever.vectorstore.add_documents(child_docs)
retriever.vectorstore.add_documents(summary_docs)

print("Multi-vector search index ready.")

In [ ]:
# OPTIONAL: look at the raw search hits, then at the parents the retriever swaps them for
peek_query = "What is the computational complexity per layer of a self-attention mechanism compared to a recurrent layer?"

raw_hits = vectorstore.similarity_search(peek_query, k=2)
for i, hit in enumerate(raw_hits):
    print(f"RAW HIT {i + 1}: {len(hit.page_content)} chars | doc_id {hit.metadata['doc_id'][:16]} | {hit.page_content[:80]!r}")

parents = retriever.invoke(peek_query)
for i, parent in enumerate(parents):
    print(f"PARENT {i + 1}: {len(parent.page_content)} chars | doc_id {parent.metadata['doc_id'][:16]}")

### Step 9 — Define the RAG Execution Function with Explainability
The prompt asks for two sections: the answer, then a trace of which sources were used.

In [ ]:
qa_template = """
You are a technical assistant. Answer the question using ONLY the provided context.

Context:
{context}

Question: {question}

Respond in exactly this format:

### Final Answer
<a clear, direct answer to the question>

### AI Tracing & Explainability
<Explain step by step how you arrived at the answer. Refer to the sources you used by their label (e.g. "Source 1"), state what each one contributed, and why it was relevant. Do not copy large blocks of text from the context — explain in your own words. Use as many lines as you need.>
"""

qa_prompt = ChatPromptTemplate.from_template(qa_template)

Those "Source 1", "Source 2" labels are what the trace refers back to, so they get attached here, before the chain is assembled.

In [ ]:
def format_docs(docs):
    """Label each source so the LLM can reference it by number in its tracing."""
    formatted = []
    for i, doc in enumerate(docs):
        parent_id = doc.metadata.get('doc_id', 'N/A')
        formatted.append(f"[Source {i+1}] (Parent ID: {parent_id})\n{doc.page_content}")
    return "\n\n".join(formatted)

Now wire retrieval and generation together: `RunnableParallel` runs the retriever and carries the raw question forward at the same time.

In [ ]:
retrieval_chain = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
)

# assign() reformats context in place while keeping the question key alongside
generation_chain = (
    RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
    | qa_prompt
    | llm
    | StrOutputParser()
)

# .assign attaches the generated answer without dropping the retrieved context
rag_chain = retrieval_chain.assign(answer=generation_chain)

### Step 10 — Execute Query

In [ ]:
def run_rag_pipeline(query: str):
    """
    Executes the RAG pipeline and displays the LLM's own answer + reasoning.
    """
    response = rag_chain.invoke(query)
    display(Markdown(response["answer"]))
    return response

In [ ]:
query = "What is the computational complexity per layer of a self-attention mechanism compared to a recurrent layer?"
response = run_rag_pipeline(query)